# CIFAR-100-C corruption robustness for the softmax baselines

Issue #6. Notebook 11 froze a temperature (T = 0.8628) and two 95%-target
abstention thresholds on CIFAR-100 calibration data. This notebook evaluates
the plain-softmax and temperature-scaled ConvNeXt baselines on CIFAR-100-C:
the same 100 classes under 15 corruptions at several severities.

Discipline: T and both thresholds are reused exactly as frozen in notebook
05 (thresholds are recomputed below with notebook 11's exact rule on the
same calibration npz, so they match bit-for-bit). Nothing is fit on
corrupted data — corruptions are scored, never calibrated on.

Scope: one corruption/severity slice is timed first; the notebook then runs
the largest of {15x{1,3,5}, 15x{3}} fitting a ~60-minute budget. The choice
and the timing evidence are printed and stored in metrics_13_cifar100c.json.

Reproduce:

```bash
python3 -m jupyter nbconvert --to notebook --execute \
  notebooks/13_cifar100_corruption_robustness.ipynb \
  --output /tmp/07_executed.ipynb
```

Override the run directory with `SDM_RUN_DIR` if your notebook-02 outputs
live elsewhere. The first run downloads CIFAR-100-C into `SDM_DATA_DIR`;
a pre-populated `CIFAR-100-C/` directory with the official layout is used
as-is (check `SOURCE.json` there for provenance when present).



In [ ]:
from __future__ import annotations

import json
import os
import tarfile
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sys
HELPER_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks" / "shift_metrics.py").exists() else Path.cwd()
sys.path.insert(0, str(HELPER_DIR))
from shift_metrics import threshold_for_target as calibrated_threshold, artifact_identity, verify_identity, metric_functions
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny
from torchvision.transforms import InterpolationMode

DATA_DIR = Path(
    os.environ.get("SDM_DATA_DIR", Path.home() / ".cache" / "sdm-vision-reliability")
).expanduser()
RUN_DIR = Path(
    os.environ.get(
        "SDM_RUN_DIR",
        Path.home()
        / ".cache"
        / "sdm-vision-reliability"
        / "runs"
        / "convnext_tiny_imagenet1k_v1"
        / "seed_42"
        / "benchmark",
    )
).expanduser()

# Frozen in notebook 05. The temperature comes from its metrics file; the
# thresholds are recomputed below with notebook 05's exact rule on the same
# calibration outputs, so they equal the frozen values bit-for-bit.
NB05 = json.loads((RUN_DIR / "metrics_05_baselines.json").read_text())
verify_identity(RUN_DIR, NB05.get("artifact_identity"))
TEMPERATURE = float(NB05["temperature"])
MIN_ADMITTED = 100
TARGET_ACCEPTED_ACCURACY = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(42)
print(f"device: {DEVICE}; frozen T = {TEMPERATURE:.4f}")


def softmax_with_temperature(logits: np.ndarray, temperature: float) -> np.ndarray:
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be finite and positive")
    logits = np.asarray(logits, dtype=np.float64)
    scaled = logits / temperature
    shifted = scaled - scaled.max(axis=1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=1, keepdims=True)


def threshold_for_target(confidences, correct, target):
    return calibrated_threshold(confidences, correct, target, min_admitted=MIN_ADMITTED)



cal = np.load(RUN_DIR / "calibration_outputs_full.npz")
cal_logits, cal_targets = cal["logits"], cal["targets"]

cal_raw = softmax_with_temperature(cal_logits, 1.0)
cal_scaled = softmax_with_temperature(cal_logits, TEMPERATURE)
cal_conf = {"softmax": cal_raw.max(axis=1), "temp-scaled": cal_scaled.max(axis=1)}
cal_ok = {
    "softmax": cal_raw.argmax(axis=1) == cal_targets,
    "temp-scaled": cal_scaled.argmax(axis=1) == cal_targets,
}
THRESHOLDS = {
    name: threshold_for_target(cal_conf[name], cal_ok[name], TARGET_ACCEPTED_ACCURACY)
    for name in ("softmax", "temp-scaled")
}
for name in THRESHOLDS:
    assert THRESHOLDS[name] == NB05["thresholds"].get(name), f"Frozen threshold drift: {name}"
# Evaluation labels are accessed only after checking the frozen calibration rules.
test = np.load(RUN_DIR / "test_outputs_full.npz")
test_logits, test_targets = test["logits"], test["targets"]
test_raw = softmax_with_temperature(test_logits, 1.0)
test_scaled = softmax_with_temperature(test_logits, TEMPERATURE)
print(f"frozen thresholds: softmax={THRESHOLDS['softmax']!r}, "
      f"temp-scaled={THRESHOLDS['temp-scaled']!r}")

# Clean reference, straight from notebook 02/05's frozen test outputs.
test_pred = test_raw.argmax(axis=1)  # argmax is identical under temperature scaling
CLEAN = {"accuracy": float((test_pred == test_targets).mean())}
for _name, _probs in (("softmax", test_raw), ("temp-scaled", test_scaled)):
    CLEAN[_name] = float((_probs.max(axis=1) >= (THRESHOLDS[_name] if THRESHOLDS[_name] is not None else np.inf)).mean())
print(f"clean test: acc={CLEAN['accuracy']:.4f} "
      f"admit_softmax={CLEAN['softmax']:.4f} admit_scaled={CLEAN['temp-scaled']:.4f}")



In [ ]:
# Same architecture and weights as notebook 02's benchmark run.
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = convnext_tiny(weights=None)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, 100)
checkpoint = torch.load(
    RUN_DIR / "best_checkpoint.pt", map_location="cpu", weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"loaded checkpoint (best val: {checkpoint['best_validation_accuracy']:.2%})")


In [ ]:
# The 15 benchmark corruptions, same order as the feature/5 CIFAR100C loader.
CORRUPTIONS = (
    "gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur",
    "glass_blur", "motion_blur", "zoom_blur", "snow", "frost", "fog",
    "brightness", "contrast", "elastic_transform", "pixelate", "jpeg_compression",
)
CIFAR100C_URL = "https://zenodo.org/record/3555552/files/CIFAR-100-C.tar"

archive = DATA_DIR / "CIFAR-100-C.tar"
target_dir = DATA_DIR / "CIFAR-100-C"
if not (target_dir / "labels.npy").exists():
    if not archive.exists():
        print(f"downloading {CIFAR100C_URL} (~2.9 GB) ...")
        urllib.request.urlretrieve(CIFAR100C_URL, archive)
    import hashlib
    digest = hashlib.md5()
    with archive.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    if digest.hexdigest() != "11f0ed0f1191edbf9fa23466ae6021d3":
        raise ValueError("Corruption archive checksum mismatch; prepare the verified cache before rerunning")
    print("extracting CIFAR-100-C ...")
    with tarfile.open(archive) as tar:
        tar.extractall(DATA_DIR, filter="data")
missing = [c for c in CORRUPTIONS if not (target_dir / f"{c}.npy").exists()]
assert not missing, f"missing corruption files: {missing}"
DATA_SOURCE = (
    json.loads((target_dir / "SOURCE.json").read_text())
    if (target_dir / "SOURCE.json").exists()
    else {"source": "local corruption cache; no SOURCE.json provenance available"}
)
c100c_labels = np.load(target_dir / "labels.npy", allow_pickle=False)
print(f"CIFAR-100-C ready: {target_dir} "
      f"labels={c100c_labels.shape} range=({c100c_labels.min()},{c100c_labels.max()})")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class CIFAR100C(Dataset):
    """One corruption/severity slice of CIFAR-100-C (10k images).

    Mirrors the feature/5 loader's validation: uint8 (50000, 32, 32, 3)
    memmap, 10k or 50k integer labels in [0, 99]. The transform is
    notebook 02/06's Resize(256)+CenterCrop(224)+ImageNet-norm eval chain.
    """

    def __init__(self, root, *, corruption, severity, transform):
        if corruption not in CORRUPTIONS:
            raise ValueError(f"Unknown corruption {corruption!r}")
        if not isinstance(severity, int) or severity not in range(1, 6):
            raise ValueError("severity must be an integer from 1 to 5")
        directory = Path(root) / "CIFAR-100-C"
        self.images = np.load(
            directory / f"{corruption}.npy", mmap_mode="r", allow_pickle=False
        )
        labels = np.load(directory / "labels.npy", allow_pickle=False)
        if self.images.shape != (50000, 32, 32, 3) or self.images.dtype != np.uint8:
            raise ValueError("Corruption images must be uint8 (50000, 32, 32, 3)")
        if labels.shape not in ((10000,), (50000,)):
            raise ValueError("labels.npy must contain 10000 or 50000 labels")
        if not np.issubdtype(labels.dtype, np.integer):
            raise ValueError("labels.npy must contain integer labels")
        if labels.min() < 0 or labels.max() > 99:
            raise ValueError("CIFAR-100-C labels must be between 0 and 99")
        self.offset = (severity - 1) * 10000
        self.targets = (
            labels if len(labels) == 10000
            else labels[self.offset:self.offset + 10000]
        )
        self.transform = transform

    def __len__(self):
        return 10000

    def __getitem__(self, index):
        if not 0 <= index < len(self):
            raise IndexError(index)
        image = Image.fromarray(self.images[self.offset + index])
        return self.transform(image), int(self.targets[index])


slice_check = CIFAR100C(
    DATA_DIR, corruption="gaussian_noise", severity=1, transform=eval_transform
)
x0, y0 = slice_check[0]
print(f"slice check: len={len(slice_check)} sample={tuple(x0.shape)} label={y0}")


In [ ]:
@torch.inference_mode()
def run_slice(corruption: str, severity: int, batch_size: int = 128):
    ds = CIFAR100C(
        DATA_DIR, corruption=corruption, severity=severity, transform=eval_transform
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)
    logit_chunks, target_chunks = [], []
    for images, targets in loader:
        logit_chunks.append(model(images.to(DEVICE)).detach().to("cpu").numpy())
        target_chunks.append(np.asarray(targets))
    return np.concatenate(logit_chunks), np.concatenate(target_chunks)


# Time ONE corruption/severity slice end to end (10k images), then run the
# largest of {15x{1,3,5}, 15x{3}} fitting the ~60-minute budget. The probe
# slice is kept and reused below, not re-run.
t0 = time.perf_counter()
probe_logits, probe_targets = run_slice("gaussian_noise", 3)
probe_s = time.perf_counter() - t0
print(f"timed one slice (gaussian_noise s3, 10k images): {probe_s:.1f}s")

BUDGET_S = 3600.0
est_full = probe_s * 45  # 15 corruptions x severities {1, 3, 5}
est_mid = probe_s * 15  # 15 corruptions x severity {3}
if est_full <= 0.85 * BUDGET_S:
    SEVERITIES, SCOPE_NAME = (1, 3, 5), "15x{1,3,5}"
else:
    SEVERITIES, SCOPE_NAME = (3,), "15x{3}"
SCOPE = {
    "name": SCOPE_NAME,
    "severities": list(SEVERITIES),
    "probe_slice": "gaussian_noise/3",
    "probe_seconds": probe_s,
    "est_full_45_slices_s": est_full,
    "est_mid_15_slices_s": est_mid,
    "budget_s": BUDGET_S,
}
print(f"estimate 15x{{1,3,5}}: {est_full / 60:.0f} min; "
      f"15x{{3}}: {est_mid / 60:.0f} min -> scope {SCOPE_NAME}")


In [ ]:
RESULTS = {("gaussian_noise", 3): (probe_logits, probe_targets)}
t_start = time.perf_counter()
for corruption in CORRUPTIONS:
    for severity in SEVERITIES:
        if (corruption, severity) in RESULTS:
            continue
        RESULTS[(corruption, severity)] = run_slice(corruption, severity)
    print(f"{corruption}: done ({(time.perf_counter() - t_start) / 60:.1f} min elapsed)")
print(f"ran {len(RESULTS)} slices in {(time.perf_counter() - t_start) / 60:.1f} min")

out_path = RUN_DIR / "cifar100c_outputs_07.npz"
np.savez_compressed(
    out_path,
    **{f"{c}__s{s}__logits": RESULTS[(c, s)][0]
       for c in CORRUPTIONS for s in SEVERITIES},
    **{f"{c}__s{s}__targets": RESULTS[(c, s)][1]
       for c in CORRUPTIONS for s in SEVERITIES},
)
print(f"saved {out_path}")


In [ ]:
def summarize(logits: np.ndarray, targets: np.ndarray) -> dict:
    raw = softmax_with_temperature(logits, 1.0)
    scaled = softmax_with_temperature(logits, TEMPERATURE)
    pred = raw.argmax(axis=1)  # temperature scaling preserves argmax
    summary = {"accuracy": float((pred == targets).mean())}
    for name, probs in (("softmax", raw), ("temp-scaled", scaled)):
        admitted = probs.max(axis=1) >= (THRESHOLDS[name] if THRESHOLDS[name] is not None else np.inf)
        summary[name] = {
            "admission_rate": float(admitted.mean()),
            "accepted_accuracy": (
                float((pred[admitted] == targets[admitted]).mean())
                if admitted.any() else float("nan")
            ),
        }
    return summary


METRICS = {(c, s): summarize(*RESULTS[(c, s)]) for c in CORRUPTIONS for s in SEVERITIES}

print(f"{'corruption':<17}{'sev':>4}{'acc':>8}{'admit_soft':>11}{'admit_scaled':>13}"
      f"{'accpt_soft':>11}{'accpt_scaled':>13}")
for c in CORRUPTIONS:
    for s in SEVERITIES:
        m = METRICS[(c, s)]
        print(f"{c:<17}{s:>4}{m['accuracy']:>8.4f}"
              f"{m['softmax']['admission_rate']:>11.4f}"
              f"{m['temp-scaled']['admission_rate']:>13.4f}"
              f"{m['softmax']['accepted_accuracy']:>11.4f}"
              f"{m['temp-scaled']['accepted_accuracy']:>13.4f}")

mean_acc = float(np.mean([METRICS[k]["accuracy"] for k in METRICS]))
mean_admit = {
    n: float(np.mean([METRICS[k][n]["admission_rate"] for k in METRICS]))
    for n in ("softmax", "temp-scaled")
}
print(f"\nclean: acc={CLEAN['accuracy']:.4f} "
      f"admit_soft={CLEAN['softmax']:.4f} admit_scaled={CLEAN['temp-scaled']:.4f}")
print(f"mean corrupted ({SCOPE_NAME}): acc={mean_acc:.4f} "
      f"admit_soft={mean_admit['softmax']:.4f} admit_scaled={mean_admit['temp-scaled']:.4f}")


In [ ]:
sev_mean_acc = [
    float(np.mean([METRICS[(c, s)]["accuracy"] for c in CORRUPTIONS]))
    for s in SEVERITIES
]
sev_mean_admit = {
    n: [float(np.mean([METRICS[(c, s)][n]["admission_rate"] for c in CORRUPTIONS]))
        for s in SEVERITIES]
    for n in ("softmax", "temp-scaled")
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.plot([0, *SEVERITIES], [CLEAN["accuracy"], *sev_mean_acc], marker="o")
ax.set_title("Mean accuracy vs corruption severity")
ax.set_xlabel("severity (0 = clean)")
ax.set_ylabel("accuracy")
ax.set_xticks([0, *SEVERITIES])
ax.set_ylim(0, 1)
ax = axes[1]
for name in ("softmax", "temp-scaled"):
    ax.plot([0, *SEVERITIES], [CLEAN[name], *sev_mean_admit[name]],
            marker="o", label=name)
ax.set_title("Mean admission rate vs corruption severity")
ax.set_xlabel("severity (0 = clean)")
ax.set_ylabel("admission rate")
ax.set_xticks([0, *SEVERITIES])
ax.set_ylim(0, 1)
ax.legend()
fig.tight_layout()
fig_path = RUN_DIR / "figures_07_cifar100c.png"
fig.savefig(fig_path, dpi=120)
print(f"saved {fig_path}")

metrics = {
    "temperature": TEMPERATURE,
    "thresholds": THRESHOLDS,
    "fit_on": "CIFAR-100 calibration only (notebook 05); nothing fit on corrupted data",
    "data_source": DATA_SOURCE,
    "scope": SCOPE,
    "clean": CLEAN,
    "mean_corrupted": {
        "accuracy": mean_acc,
        **{f"admission_rate_{n}": mean_admit[n] for n in mean_admit},
    },
    "per_slice": {
        f"{c}/s{s}": METRICS[(c, s)] for c in CORRUPTIONS for s in SEVERITIES
    },
}
metrics_path = RUN_DIR / "metrics_07_cifar100c.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
print(f"saved {metrics_path}")


## Reading the results

What to look for:

- Accuracy is identical for both baselines: temperature scaling (T < 1)
  sharpens probabilities but preserves the argmax. The two methods differ
  only in admission rate — the temp-scaled threshold (0.85) sits higher
  than the plain-softmax one (0.75), so compare the admission columns.
- Accepted accuracy near the 95% calibration target on mild corruptions
  means the frozen abstention rule degrades gracefully; accepted accuracy
  collapsing toward raw accuracy means confidence stops being selective
  and the rule admits mostly by luck.
- If temperature scaling admits more corrupted images than plain softmax
  at similar accepted accuracy, the sharper probabilities buy calibration
  on clean CIFAR-100 at the cost of louder mistakes under shift — worth
  naming in the SDM comparison, since SDM is supposed to stay quiet
  off-manifold.

